In [2]:
import pandas as pd
import numpy as np
from validphys.loader import FallbackLoader as Loader
from validphys.api import API
from collections import defaultdict
import os
from nnpdf_data import legacy_to_new_map
import matplotlib.pyplot as plt

In [3]:
tcm_fitname = "2026-05-04_ap_nnlo_thr3_mc_var_mhou_pc"

In [4]:
fit = API.fit(fit=tcm_fitname)
# find the name of point prescription mc_pp
pps = fit.as_input()["theorycovmatconfig"]["point_prescriptions"]
mc_pp_id, mc_pp = [[j,i] for j,i in enumerate(pps) if "mc" in i][0]
# build validphys inputs
common_dict = dict(
    dataset_inputs={"from_": "fit"},
    fit=fit.name,
    fits=[fit.name],
    use_cuts="fromfit",
    metadata_group="nnpdf31_process",
)
theoryids_dict = ({
        "point_prescription": mc_pp,
        "theoryid": {"from_": "theory"},
        "theory": {"from_": "fit"},
        "theorycovmatconfig": {"from_": "fit"},
    } | common_dict)
# find theoryids
theoryids = API.theoryids(**theoryids_dict)

In [5]:
# goup theories by mc value. MAY NEED ALTERING
theory_plus = theoryids[2].id
theory_mid = theoryids[0].id
theory_min = theoryids[1].id

In [6]:
thcov_input_pdf = fit.as_input()["theorycovmatconfig"]["pdf"]


# Inputs for central theory (used to construct the alphas covmat)
inps_central = dict(theoryid=theory_mid, pdf=thcov_input_pdf, **common_dict)

# Inputs for plus theory (used to construct the alphas covmat)
inps_plus = dict(theoryid=theory_plus, pdf=thcov_input_pdf, **common_dict)

# Inputs for minus theory prediction (used to construct the alphas covmat)
inps_minus = dict(theoryid=theory_min, pdf=thcov_input_pdf, **common_dict)

# inputs for the computation of the prediction of the fit with cov=C+S, where S
# is computed using the inps_central, inps_plus, and inps_minus dictionaries
inps_central_fit = dict(theoryid=theory_mid, pdf={"from_": "fit"}, **common_dict)

# find priors
prior_theorypreds_central = API.group_result_central_table_no_table(**inps_central)["theory_central"]
prior_theorypreds_plus = API.group_result_central_table_no_table(**inps_plus)["theory_central"]
prior_theorypreds_minus = API.group_result_central_table_no_table(**inps_minus)["theory_central"]

# find the values of mc
mc_plus = API.theory_info_table(theory_db_id=theory_plus).loc["mc"].iloc[0]
mc_central = API.theory_info_table(theory_db_id=theory_mid).loc["mc"].iloc[0]
mc_min = API.theory_info_table(theory_db_id=theory_min).loc["mc"].iloc[0]

# make sure the mc shift in both directions is symmetric
delta_mc_plus = mc_plus - mc_central
delta_mc_min = mc_central - mc_min
if abs(delta_mc_min - delta_mc_plus) > 1e-6:
    raise ValueError("alphas shifts in both directions is not symmetric")
else:
    mc_step_size = delta_mc_min

# get the covmat scaling factor, it should be 1 in our case
covmat_scaling_factor = fit.as_input().get("theorycovmatconfig",{}).get("rescale_alphas_covmat",1.0)

# the below seem to be all priors
beta_tilde = np.sqrt(1) * (mc_step_size / np.sqrt(2)) * np.array([1, -1])
S_tilde = beta_tilde @ beta_tilde

delta_plus = (np.sqrt(covmat_scaling_factor) / np.sqrt(2)) * (
    prior_theorypreds_plus - prior_theorypreds_central
)
delta_minus = (np.sqrt(covmat_scaling_factor) / np.sqrt(2)) * (
    prior_theorypreds_minus - prior_theorypreds_central
)

beta = [delta_plus, delta_minus]
S_hat = pd.Series(beta_tilde @ beta, index=delta_minus.index)

S = np.outer(delta_plus, delta_plus) + np.outer(delta_minus, delta_minus)
S = pd.DataFrame(S, index=delta_minus.index, columns=delta_minus.index)

# read the fit covmat
stored_mc_covmat = pd.read_csv(
    fit.path / f"tables/datacuts_theory_theorycovmatconfig_point_prescriptions{mc_pp_id}_theory_covmat_custom_per_prescription.csv",
    index_col=[0, 1, 2],
    header=[0, 1, 2],
    sep="\t|,",
    encoding="utf-8",
    engine="python",
).fillna(0)


storedcovmat_index = pd.MultiIndex.from_tuples(
    [(aa, bb, np.int64(cc)) for aa, bb, cc in stored_mc_covmat.index],
    names=["group", "dataset", "id"],
)  # make sure theoryID is an integer, same as in S

# format the covmat from fit
stored_mc_covmat = pd.DataFrame(
    stored_mc_covmat.values, index=storedcovmat_index, columns=storedcovmat_index
)
new_names = {d[0]: legacy_to_new_map(d[0])[0] for d in stored_mc_covmat.index}
stored_mc_covmat.rename(columns=new_names, index=new_names, level=1, inplace=True) # rename datasets using the legacy to new map
stored_mc_covmat = stored_mc_covmat.reindex(S.index).T.reindex(S.index)

# compare the covmats
if not np.allclose(S, stored_mc_covmat):
    print("Reconstructed theory covmat, S, is not the same as the stored covmat!")

# results per replica
theorypreds_fit = API.group_result_table_no_table(**inps_central_fit).iloc[:, 2:]

# experimental covmat
exp_covmat = API.groups_covmat(
    use_t0=True,
    datacuts={"from_": "fit"},
    t0pdfset={"from_": "datacuts"},
    theoryid= {"from_": "theory"},
    theory={"from_": "fit"},
    **common_dict
)

# theory covmat
total_th_covmat = pd.read_csv(
    fit.path / f"tables/datacuts_theory_theorycovmatconfig_theory_covmat_custom.csv",
    index_col=[0, 1, 2],
    header=[0, 1, 2],
    sep="\t|,",
    encoding="utf-8",
    engine="python",
).fillna(0)

# reindex the theory covmat
new_names = {d[0]: legacy_to_new_map(d[0])[0] for d in total_th_covmat.index}
total_th_covmat.rename(columns=new_names, index=new_names, level=1, inplace=True) # rename datasets using the legacy to new map
total_th_covmat_index = pd.MultiIndex.from_tuples(
    [(aa, bb, np.int64(cc)) for aa, bb, cc in total_th_covmat.index],
    names=["group", "dataset", "id"],
) # make sure the index is an int, just as it is in S
total_th_covmat = pd.DataFrame(
    total_th_covmat.values, index=total_th_covmat_index, columns=total_th_covmat_index
)
total_th_covmat = total_th_covmat.reindex(S.index).T.reindex(S.index)

# mean prediction != prediction of mean PDF
mean_prediction = theorypreds_fit.mean(axis=1)

# calculate covariance matrix over replicas
X = np.zeros_like(S.values)
for i in range(theorypreds_fit.shape[1]):
    X += np.outer(
        (theorypreds_fit.iloc[:, i] - mean_prediction),
        (theorypreds_fit.iloc[:, i] - mean_prediction),
    )
X *= 1 / theorypreds_fit.shape[1]
X = pd.DataFrame(X, index=theorypreds_fit.index, columns=theorypreds_fit.index)

# In the computation we use <D>_rep and not the central value of the data D_exp, though if
# resample_negative_pseudodata: false
# is set in the n3fit runcard, D_exp and <D>_rep should be the same as N_rep -> inf.
pseudodata = API.read_pdf_pseudodata(**common_dict)
dat_reps = pd.concat(
    [i.pseudodata for i in pseudodata], axis=1
)
# pseudodata values per each replica
dat_reps = dat_reps.reindex(S.index)


# dat_central = API.group_result_central_table_no_table(**inps_central)["data_central"]
# mean pseudodata (should be close to data)
dat_central = dat_reps.mean(axis=1)

# inverse of the full covmat
invcov = pd.DataFrame(np.linalg.inv(exp_covmat + total_th_covmat),index=exp_covmat.index, columns=exp_covmat.index)
invcov = invcov.reindex(S.index).T.reindex(S.index)

# delta_T_tilde is Eq. 3.37 in https://arxiv.org/pdf/2105.05114
# this is the shift in m_c
delta_T_tilde = -S_hat @ invcov @ (mean_prediction - dat_central)

/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: divide by zero encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: overflow encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: invalid value encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: divide by zero encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: overflow encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/vali

LHAPDF 6.5.5 loading /opt/miniconda3/envs/nnpdf_dev/share/LHAPDF/NNPDF40_nnlo_as_01180/NNPDF40_nnlo_as_01180_0000.dat
NNPDF40_nnlo_as_01180 PDF set, member #0, version 1; LHAPDF ID = 331100


/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: divide by zero encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: overflow encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: invalid value encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: divide by zero encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: overflow encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/vali

LHAPDF 6.5.5 loading all 91 PDFs in set 2026-05-04_ap_nnlo_thr3_mc_var_mhou_pc
2026-05-04_ap_nnlo_thr3_mc_var_mhou_pc, version 1; 91 PDF members
LHAPDF 6.5.5 loading /opt/miniconda3/envs/nnpdf_dev/share/LHAPDF/240701-02-rs-nnpdf40-baseline/240701-02-rs-nnpdf40-baseline_0000.dat
240701-02-rs-nnpdf40-baseline PDF set, member #0, version 1


/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: divide by zero encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: overflow encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: invalid value encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: divide by zero encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: overflow encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmat

In [ ]:
# delta_T_tilde_reps needs to be approximated to get X_hat and X_tilde

# get data_theory central values
data_theory_results =  API.group_result_central_table_no_table(**inps_central)

# sort out the central theory predictions
data_central = data_theory_results["data_central"]
data_central.reindex(theorypreds_fit.index)
data_central_reps = pd.DataFrame(data_central)
for i in range(1,91):
    if i <10:
        data_central_reps[f"rep_0000{i}"] = data_central
    else:
        data_central_reps[f"rep_000{i}"] = data_central
remove_first_col = data_central_reps.pop("data_central")

# Jaco's code to approximate delta_T_tilde_reps
delta_T_tilde_reps = -S_hat @ invcov @ theorypreds_fit.sub(data_central_reps)

noise = np.random.normal(loc=0.0, scale=1.0, size =(1, delta_T_tilde_reps.shape[0]))
stochastic_shift = (S_tilde-S_hat @ invcov @ S_hat.T) * noise
# delta_T_tilde_reps += stochastic_shift[0]

X_tilde = 0
for i in range(delta_T_tilde_reps.shape[0]):
    X_tilde += delta_T_tilde_reps.iloc[i]*delta_T_tilde_reps.iloc[i]
X_tilde *= 1 / delta_T_tilde_reps.shape[0]

X_hat = np.zeros_like(S_hat.T)
for i in range(delta_T_tilde_reps.shape[0]):
    X_hat += delta_T_tilde_reps.iloc[i] * (theorypreds_fit.iloc[:, i] - mean_prediction)
X_hat *= 1 / delta_T_tilde_reps.shape[0]

# P_tilde is Eq. 3.38.
# This is the variance. Need to include all the terms
P_tilde_partial = S_hat.T @ invcov @ X @ invcov @ S_hat + S_tilde - S_hat.T @ invcov @ S_hat

# Calculate the missing terms
missing_terms = X_tilde - 2 * X_hat @ invcov @ S_hat

# full P_tilde, probably wrong
P_tilde = missing_terms + P_tilde_partial

# the result
pred = mc_central + delta_T_tilde
unc = np.sqrt(P_tilde_partial)
print(rf"Prediction for $mc$: {pred:.5f} ± {unc:.5f}")

# replica distribution
tcm_vals = mc_central -S_hat @ invcov @ (theorypreds_fit.to_numpy() - dat_central.reindex(S.index).to_numpy().reshape(-1,1))

/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: divide by zero encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: overflow encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats.py:220: RuntimeWarning: invalid value encountered in matmul
  covmat = diag + special_sys.to_numpy() @ special_sys.to_numpy().T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: divide by zero encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/validphys/covmats_utils.py:88: RuntimeWarning: overflow encountered in matmul
  return np.diag(diagonal) + corr_sys_mat @ corr_sys_mat.T
/Users/s2850353/Documents/nnpdf/validphys2/src/vali

In [ ]:
from scipy.stats import norm


bins = np.linspace(1, 2, 31)
plt.hist(tcm_vals, bins=bins, density=True, alpha=0.6, label='TCM', color="C0")

x = np.linspace(bins[0], bins[-1], 1000)
mu_tcm, std_tcm = np.mean(tcm_vals), np.std(tcm_vals)
plt.plot(x, norm.pdf(x, mu_tcm, std_tcm), 'b-', lw=2, color="C0", alpha=0.6)


# Add labels and legend
plt.xlabel(r'$m_\text{charm}$')
plt.ylabel('normalized frequency')
# title = "with positivity" if positivity else "without positivity"
plt.title(r"NNLO$_\mathrm{QCD}$+MHOU")
plt.tight_layout()
plt.legend()
# plt.savefig(f"../plots/crm_vs_tcm_hist_{filename}.pdf")
plt.show()